In [12]:
import math
import matplotlib.pyplot as plt
import networkx as nx
import pandas as pd
import os
import heapq
from collections import Counter

In [2]:
PATIENT_ID = "../data/PatientIds.txt"
NETWORK_FOLDER = "../data/NetworksFromUp-InBetween1/"

In [3]:
# read patients lists (primary, secondary, and all for primary)
patient_ids = pd.read_csv(PATIENT_ID, header=None).iloc[:, 0].tolist()

In [4]:
# read the networks made for each case
patient_files = {f"{pid}.txt" for pid in patient_ids}

dict_of_graphs = {
    file: nx.read_edgelist(
        os.path.join(NETWORK_FOLDER, file),
        delimiter=";",
        nodetype=str,
        create_using=nx.DiGraph()
    )
    for file in os.listdir(NETWORK_FOLDER)
    if file in patient_files
}

In [5]:
#calculate network properties & centralities
dict_of_results = {}

for key, graph in dict_of_graphs.items():
    if graph.number_of_nodes() == 0:
        dict_of_results[key] = {}
        continue

    # find the largest strongly connected component
    largest_set = max(nx.strongly_connected_components(graph), key=len)
    cc = graph.subgraph(largest_set).copy()

    # in-degree and out-degree
    indeg = nx.in_degree_centrality(cc)
    outdeg = nx.out_degree_centrality(cc)

    # eigenvector
    try:
        eign = nx.eigenvector_centrality_numpy(cc)
    except Exception:
        eign = {}

    # betweenness
    between = nx.betweenness_centrality(cc)

    # shortest paths
    sp = dict(nx.all_pairs_shortest_path_length(cc))
    
    # closeness
    close = nx.closeness_centrality(cc, distance=None)

    # harmonic
    harmonic = nx.harmonic_centrality(cc)

    # eccentricity-related metrics
    ecc = nx.eccentricity(cc, sp=sp)
    diam = max(ecc.values())
    radius = min(ecc.values())

    # average shortest path length
    avshpath = sum(
        d for paths in sp.values() for d in paths.values()
    ) / (cc.number_of_nodes() * (cc.number_of_nodes() - 1))

    density = nx.density(cc)
    avgclustering = nx.average_clustering(cc)

    # top-5
    #top5 = lambda d: heapq.nlargest(5, d.items(), key=lambda x: x[1])
    top5 = lambda d: [k for k, v in heapq.nlargest(5, d.items(), key=lambda x: x[1])]
    
    dict_of_results[key] = {
        "in_degree": top5(indeg),
        "out_degree": top5(outdeg),
        "eigenvector": top5(eign),
        "closeness": top5(close),
        "betweenness": top5(between),
        "harmonic": top5(harmonic),
        "eccentricity": top5(ecc),
        "diameter": diam,
        "radius": radius,
        "average_shortest_path_length": avshpath,
        "density": density,
        "avgclustering": avgclustering,
    }

In [6]:
df = pd.DataFrame(data=dict_of_results)
df_tr=df.transpose()
df_tr.to_excel('../data/topology.xlsx')

In [11]:
# find the most frequent hubs
all_elements = []

columns_to_check = ['in_degree', 'out_degree','eigenvector', 'closeness', 'betweenness', 'harmonic', 'eccentricity']

for col in columns_to_check:
    for cell in df_tr[col].dropna():
        all_elements.extend(cell)
        
top_20 = Counter(all_elements).most_common(20)

print(top_20)

[('SRC', 906), ('TP53', 755), ('UBC', 714), ('STAT3', 485), ('CTNNB1', 303), ('GSK3B', 302), ('MAPK1', 246), ('EGFR', 198), ('PTK2', 160), ('CDK1', 151), ('CDK2', 151), ('STXBP2', 110), ('VAMP4', 108), ('PRKCA', 107), ('NUP50', 100), ('STX11', 76), ('KIF5C', 67), ('MAPK14', 52), ('SSRP1', 48), ('SUPT6H', 48)]
